In [1]:
%pip install pandas openpyxl


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
from slot_util import load_slot_tables, print_df_attrs, get_reel_symbols, get_reelstop_df, add_wild_combination_counts, get_line_with_wild_pattern_df, get_sticky_wild_state_df, get_sticky_wild_line_probability_df, get_sticky_wild_state_probability_df

In [3]:
reel_df, paytable_df, _, game_settings = load_slot_tables("sheet/StickyWild.xlsx")

In [4]:
print_df_attrs(reel_df)
display(reel_df)

print_df_attrs(paytable_df)
display(paytable_df)

print_df_attrs(game_settings)
display(game_settings)

shape: (16, 5)
表: 滾輪表
列: Stop 索引
行: 第幾個滾輪


,R1,R2,R3,R4,R5
0,W,W,W,W,W
1,K,K,K,K,K
2,Q,Q,Q,Q,Q
3,J,J,J,J,J
4,10,10,10,10,10
5,9,9,9,9,9
6,K,K,K,K,K
7,Q,Q,Q,Q,Q
8,J,J,J,J,J
9,10,10,10,10,10


shape: (6, 3)
表: 賠率表
列: 連線的 Symbol
行: 由左到右的連線數


,3,4,5
Symbol,,,
W,150,300,1000
K,5,30,100
Q,5,25,100
J,5,20,75
10,5,20,75
9,5,10,50


shape: (1,)
表: 遊戲設定
列: 設定名稱
值: 設定值


Setting
FreeSpinCount    10
Name: Value, dtype: int64

In [5]:
symbols = get_reel_symbols(reel_df)
symbols

['W', 'K', 'Q', 'J', '10', '9']

In [6]:
reelstop_df = get_reelstop_df(reel_df, symbols)

In [7]:
add_wild_combination_counts(reelstop_df, symbols)

In [8]:
print_df_attrs(reelstop_df)
display(reelstop_df)

shape: (17, 5)
表: Symbol 在各滾輪的 Stop 數
列: Symbol
行: 第幾個滾輪


,R1,R2,R3,R4,R5
Symbol,,,,,
W,1,1,1,1,1
K,3,3,3,3,3
Q,3,3,3,3,2
J,3,3,3,2,2
10,3,3,2,2,2
9,3,2,2,2,2
-,16,15,14,13,12
WK,4,4,4,4,4
WQ,4,4,4,4,3


In [9]:
sticky_state_df = get_sticky_wild_state_df(reel_df)

In [10]:
print_df_attrs(sticky_state_df)
display(sticky_state_df)

shape: (32, 6)
表: Sticky Wild 狀態
列: Sticky Wild 狀態
行.State: 0: 未黏住; 1: 已黏住
行.Ri: 第 i 輪位置是否已有 Sticky Wild


,State,R1,R2,R3,R4,R5
0,00000,0,0,0,0,0
1,00001,0,0,0,0,1
2,00010,0,0,0,1,0
3,00011,0,0,0,1,1
4,00100,0,0,1,0,0
5,00101,0,0,1,0,1
6,00110,0,0,1,1,0
7,00111,0,0,1,1,1
8,01000,0,1,0,0,0
9,01001,0,1,0,0,1


In [11]:
line_df = get_line_with_wild_pattern_df(reel_df, paytable_df, symbols)
sticky_line_df = get_sticky_wild_line_probability_df(line_df, sticky_state_df, reelstop_df, paytable_df)

In [12]:
print_df_attrs(sticky_line_df)
display(sticky_line_df)

shape: (268, 42)
表: 各 Sticky Wild 狀態的連線組合機率
列: LineType x Symbol x MatchCount
行.Pay: 該連線組合的得分
行.State: P(Line combination | Sticky Wild state)


,LineType,Symbol,MatchCount,LeadingWCount,R1,R2,R3,R4,R5,Pay,...,10110,10111,11000,11001,11010,11011,11100,11101,11110,11111
0,LineK,K,5,0,K,K,K,K,K,100,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.0
1,LineK,K,4,0,K,K,K,K,X_WK,30,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.0
2,LineK,K,3,0,K,K,K,X_WK,-,5,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.0
3,LineK,Q,5,0,Q,Q,Q,Q,Q,100,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.0
4,LineK,Q,4,0,Q,Q,Q,Q,X_WQ,25,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
263,LineWK,9,3,1,W,9,W,X_W9,-,5,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.0
264,LineWK,9,3,0,9,W,W,X_W9,-,5,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.0
265,LineW,W,5,5,W,W,W,W,W,1000,...,0.005556,0.066667,0.000458,0.005495,0.005952,0.071429,0.00641,0.076923,0.083333,1.0
266,LineW,W,4,4,W,W,W,W,S,300,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.0


In [13]:
state_cols = sticky_state_df["State"].tolist()
sticky_state_rtp = (
    sticky_line_df[state_cols]
    .mul(
        sticky_line_df["Pay"],
        axis="index",
    )
    .sum(axis="index")
    .rename("RTP")
)

In [14]:
display(
    sticky_state_rtp
    .to_frame()
    .style.format({"RTP": "{:.2%}"})
)

,RTP
00000,120.84%
00001,220.05%
00010,324.02%
00011,702.38%
00100,483.36%
00101,852.56%
00110,1269.62%
00111,2833.33%
01000,497.11%
01001,875.69%


In [15]:
sticky_state_probability_df = get_sticky_wild_state_probability_df(reelstop_df, sticky_state_df, game_settings["FreeSpinCount"])

In [16]:
print_df_attrs(sticky_state_probability_df)
display(sticky_state_probability_df.style.format("{:.4%}"))

shape: (10, 32)
表: 各 Free Spin 開始時的 Sticky Wild 狀態機率
列: 第幾把 Free Spin
行.State: P(Sticky Wild state)


,00000,00001,00010,00011,00100,00101,00110,00111,01000,01001,01010,01011,01100,01101,01110,01111,10000,10001,10010,10011,10100,10101,10110,10111,11000,11001,11010,11011,11100,11101,11110,11111
FreeSpin,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
1,100.0000%,0.0000%,0.0000%,0.0000%,0.0000%,0.0000%,0.0000%,0.0000%,0.0000%,0.0000%,0.0000%,0.0000%,0.0000%,0.0000%,0.0000%,0.0000%,0.0000%,0.0000%,0.0000%,0.0000%,0.0000%,0.0000%,0.0000%,0.0000%,0.0000%,0.0000%,0.0000%,0.0000%,0.0000%,0.0000%,0.0000%,0.0000%
2,68.7500%,6.2500%,5.7292%,0.5208%,5.2885%,0.4808%,0.4407%,0.0401%,4.9107%,0.4464%,0.4092%,0.0372%,0.3777%,0.0343%,0.0315%,0.0029%,4.5833%,0.4167%,0.3819%,0.0347%,0.3526%,0.0321%,0.0294%,0.0027%,0.3274%,0.0298%,0.0273%,0.0025%,0.0252%,0.0023%,0.0021%,0.0002%
3,47.2656%,8.9844%,8.2058%,1.5598%,7.5513%,1.4354%,1.3110%,0.2492%,6.9934%,1.3293%,1.2141%,0.2308%,1.1173%,0.2124%,0.1940%,0.0369%,6.5122%,1.2378%,1.1306%,0.2149%,1.0404%,0.1978%,0.1806%,0.0343%,0.9635%,0.1832%,0.1673%,0.0318%,0.1539%,0.0293%,0.0267%,0.0051%
4,32.4951%,9.6924%,8.8196%,2.6306%,8.0905%,2.4132%,2.1959%,0.6550%,7.4725%,2.2288%,2.0281%,0.6049%,1.8605%,0.5549%,0.5050%,0.1506%,6.9419%,2.0706%,1.8841%,0.5620%,1.7284%,0.5155%,0.4691%,0.1399%,1.5963%,0.4761%,0.4333%,0.1292%,0.3974%,0.1185%,0.1079%,0.0322%
5,22.3404%,9.3002%,8.4304%,3.5096%,7.7086%,3.2091%,2.9089%,1.2110%,7.1000%,2.9557%,2.6793%,1.1154%,2.4499%,1.0199%,0.9245%,0.3849%,6.5801%,2.7393%,2.4831%,1.0337%,2.2705%,0.9452%,0.8568%,0.3567%,2.0912%,0.8706%,0.7892%,0.3285%,0.7216%,0.3004%,0.2723%,0.1134%
6,15.3590%,8.3714%,7.5588%,4.1200%,6.8888%,3.7547%,3.3903%,1.8479%,6.3270%,3.4485%,3.1138%,1.6972%,2.8378%,1.5467%,1.3966%,0.7612%,5.8493%,3.1882%,2.8787%,1.5690%,2.6235%,1.4300%,1.2912%,0.7037%,2.4096%,1.3133%,1.1859%,0.6464%,1.0807%,0.5891%,0.5319%,0.2899%
7,10.5593%,7.2385%,6.5097%,4.4625%,5.9126%,4.0531%,3.6450%,2.4987%,5.4147%,3.7119%,3.3381%,2.2883%,3.0319%,2.0784%,1.8692%,1.2813%,4.9935%,3.4231%,3.0784%,2.1103%,2.7960%,1.9167%,1.7237%,1.1816%,2.5606%,1.7553%,1.5786%,1.0821%,1.4338%,0.9829%,0.8839%,0.6059%
8,7.2595%,6.0889%,5.4533%,4.5739%,4.9360%,4.1400%,3.7079%,3.1100%,4.5071%,3.7803%,3.3857%,2.8397%,3.0645%,2.5703%,2.3021%,1.9308%,4.1458%,3.4773%,3.1143%,2.6121%,2.8189%,2.3643%,2.1176%,1.7761%,2.5739%,2.1589%,1.9335%,1.6217%,1.7501%,1.4679%,1.3147%,1.1027%
9,4.9909%,5.0204%,4.4775%,4.5039%,4.0385%,4.0623%,3.6230%,3.6444%,3.6764%,3.6981%,3.2982%,3.3177%,2.9748%,2.9924%,2.6688%,2.6845%,3.3730%,3.3929%,3.0260%,3.0439%,2.7293%,2.7454%,2.4486%,2.4630%,2.4846%,2.4993%,2.2290%,2.2422%,2.0105%,2.0223%,1.8037%,1.8143%


In [19]:
sticky_spin_rtp_df = sticky_state_probability_df.mul(sticky_state_rtp, axis="columns")
sticky_spin_rtp_df["RTP"] = sticky_spin_rtp_df.sum(axis="columns")

In [20]:
display(
    sticky_spin_rtp_df.style.format("{:.2%}")
)

,00000,00001,00010,00011,00100,00101,00110,00111,01000,01001,01010,01011,01100,01101,01110,01111,10000,10001,10010,10011,10100,10101,10110,10111,11000,11001,11010,11011,11100,11101,11110,11111,RTP
FreeSpin,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
1,120.84%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,120.84%
2,83.07%,13.75%,18.56%,3.66%,25.56%,4.10%,5.60%,1.14%,24.41%,3.91%,5.32%,1.08%,8.77%,1.27%,1.79%,0.39%,23.75%,3.80%,5.19%,1.05%,8.54%,1.24%,1.75%,0.38%,8.22%,1.18%,1.67%,0.37%,4.18%,0.49%,0.75%,0.19%,265.15%
3,57.11%,19.77%,26.59%,10.96%,36.50%,12.24%,16.64%,7.06%,34.76%,11.64%,15.78%,6.70%,25.95%,7.88%,11.06%,5.07%,33.74%,11.29%,15.35%,6.50%,25.19%,7.63%,10.74%,4.92%,24.19%,7.29%,10.26%,4.71%,25.56%,6.30%,9.58%,5.08%,514.05%
4,39.27%,21.33%,28.58%,18.48%,39.11%,20.57%,27.88%,18.56%,37.15%,19.52%,26.37%,17.55%,43.21%,20.60%,28.78%,20.71%,35.97%,18.89%,25.59%,16.99%,41.84%,19.88%,27.90%,20.06%,40.08%,18.95%,26.56%,19.15%,65.99%,25.53%,38.65%,32.18%,901.87%
5,27.00%,20.47%,27.32%,24.65%,37.26%,27.36%,36.93%,34.31%,35.29%,25.88%,34.83%,32.37%,56.90%,37.85%,52.70%,52.92%,34.10%,24.98%,33.72%,31.26%,54.97%,36.45%,50.96%,51.12%,52.51%,34.66%,48.38%,48.69%,119.80%,64.70%,97.57%,113.36%,1461.26%
6,18.56%,18.42%,24.49%,28.94%,33.30%,32.01%,43.04%,52.36%,31.45%,30.20%,40.48%,49.25%,65.91%,57.41%,79.61%,104.67%,30.31%,29.08%,39.10%,47.44%,63.51%,55.14%,76.79%,100.87%,60.50%,52.28%,72.70%,95.80%,179.43%,126.87%,190.59%,289.90%,2220.41%
7,12.76%,15.93%,21.09%,31.34%,28.58%,34.56%,46.28%,70.80%,26.92%,32.50%,43.40%,66.40%,70.42%,77.14%,106.55%,176.18%,25.87%,31.22%,41.81%,63.81%,67.69%,73.92%,102.51%,169.37%,64.29%,69.88%,96.78%,160.39%,238.05%,211.70%,316.74%,605.93%,3200.80%
8,8.77%,13.40%,17.67%,32.13%,23.86%,35.30%,47.08%,88.12%,22.41%,33.10%,44.02%,82.40%,71.17%,95.40%,131.23%,265.49%,21.48%,31.72%,42.30%,78.99%,68.24%,91.18%,125.94%,254.57%,64.63%,85.94%,118.54%,240.36%,290.56%,316.16%,471.09%,1102.67%,4415.91%
9,6.03%,11.05%,14.51%,31.63%,19.52%,34.63%,46.00%,103.26%,18.28%,32.38%,42.88%,96.27%,69.09%,111.06%,152.14%,369.13%,17.48%,30.95%,41.10%,92.04%,66.08%,105.87%,145.62%,353.03%,62.38%,99.49%,136.66%,332.32%,333.79%,435.58%,646.31%,1814.30%,5870.86%


In [24]:
print(f"RTP: {sticky_spin_rtp_df['RTP'].sum():.2%}")

RTP: 26533.96%
